In [0]:
from pyspark.sql.functions import explode, col, to_timestamp

# Read the Bronze Delta table
bronze_df = spark.table("workspace.default.bronze_matches")

# Convert the nested matches array into individual rows
matches_df = bronze_df.select(
    explode(col("matches")).alias("match")
)

print("Total matches:", matches_df.count())
display(matches_df)


In [0]:
# Extract useful fields from each nested match
silver_df = matches_df.select(

    # Match identifiers
    col("match.id").alias("match_id"),
    col("match.matchday").alias("matchday"),
    col("match.status").alias("match_status"),

    # Match date
    to_timestamp(col("match.utcDate")).alias("match_date"),

    # Home team
    col("match.homeTeam.id").alias("home_team_id"),
    col("match.homeTeam.name").alias("home_team"),

    # Away team
    col("match.awayTeam.id").alias("away_team_id"),
    col("match.awayTeam.name").alias("away_team"),

    # Full-time score
    col("match.score.fullTime.home").alias("home_goals"),
    col("match.score.fullTime.away").alias("away_goals"),

    # Half-time score
    col("match.score.halfTime.home").alias("home_ht_goals"),
    col("match.score.halfTime.away").alias("away_ht_goals"),

    # Match outcome
    col("match.score.winner").alias("winner"),

    # Season identifier
    col("match.season.id").alias("season_id"),

    # Last updated timestamp
    to_timestamp(col("match.lastUpdated")).alias("last_updated")
)

display(silver_df)

# Remove duplicate match IDs
silver_df = silver_df.dropDuplicates(["match_id"])

# Remove records without a match ID
silver_df = silver_df.dropna(subset=["match_id"])

# Validate that match IDs are unique
total_matches = silver_df.count()
unique_matches = silver_df.select("match_id").distinct().count()

assert total_matches == unique_matches, "Duplicate match IDs found!"

# Validate that important fields are populated
assert silver_df.filter(col("home_team_id").isNull()).count() == 0
assert silver_df.filter(col("away_team_id").isNull()).count() == 0

print("Data validation passed!")
print("Total clean matches:", total_matches)

# Save the cleaned and structured matches
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_matches")
)

print("Silver matches table created successfully!")




In [0]:
%sql
SELECT
    matchday,
    home_team,
    away_team,
    home_goals,
    away_goals,
    winner
FROM workspace.default.silver_matches
ORDER BY matchday, match_date
LIMIT 20;
